In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import csv
import chromedriver_autoinstaller

In [4]:
options = Options()
options.add_argument('--start-maximized')

In [5]:
driver = webdriver.Chrome(options=options)

In [6]:
url = 'https://smartstore.naver.com/237/products/5187958641?NaPm=ct%3Dmbwkbblc%7Cci%3De68216037c7f172b6f11d28f9f329748e036c0e4%7Ctr%3Dslsl%7Csn%3D484739%7Chk%3D2136891136fe877ba3bccf8b80b5eea65ee1aa3c&nl-au=120b68102ac342d0bc46485146dde744&nl-query=xt-6'
driver.get(url)
time.sleep(5) 
wait = WebDriverWait(driver, 20)

In [6]:
from selenium.webdriver.support import expected_conditions as EC

try:
    review_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), '리뷰')]"))) 
    print("리뷰 보기 버튼 클릭 성공")
except Exception as e:
    print("리뷰 보기 버튼 클릭 실패:", e)
    driver.quit()
    exit()
time.sleep(5)

리뷰 보기 버튼 클릭 성공


In [7]:
def get_reviews():
    reviews = []
    try:
        review_elements = wait.until(EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "span._2L3vDiadT9")
        ))
        for elem in review_elements:
            text = elem.text.strip()
            if text:
                reviews.append(text)
    except Exception as e:
        print("리뷰 불러오기 실패:", e)
    return reviews

all_reviews = []
page_num = 1

while True:
    print(f"===== {page_num}페이지 리뷰 수집 중 =====")
    reviews = get_reviews()
    print(f"{page_num}페이지 리뷰 수: {len(reviews)}")
    all_reviews.extend(reviews)

    try:
        next_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.fAUKm1ewwo._2Ar8-aEUTq._nlog_click")))
        next_button_text = next_button.text.strip()
        
        if next_button_text != "다음":
            print("더 이상 다음 페이지가 없습니다.")
            break
        
        next_button.click()
        page_num += 1
        time.sleep(2) 
    except Exception as e:
        print("다음 페이지로 이동 실패 또는 마지막 페이지:", e)
        break

print("크롤링 완료")
print(f"총 수집한 리뷰 수: {len(all_reviews)}")

===== 1페이지 리뷰 수집 중 =====
,1페이지 리뷰 수: 48
,===== 2페이지 리뷰 수집 중 =====
,2페이지 리뷰 수: 77
,===== 3페이지 리뷰 수집 중 =====
,3페이지 리뷰 수: 68
,===== 4페이지 리뷰 수집 중 =====
,4페이지 리뷰 수: 62
,===== 5페이지 리뷰 수집 중 =====
,5페이지 리뷰 수: 58
,===== 6페이지 리뷰 수집 중 =====
,6페이지 리뷰 수: 40
,===== 7페이지 리뷰 수집 중 =====
,7페이지 리뷰 수: 40
,===== 8페이지 리뷰 수집 중 =====
,8페이지 리뷰 수: 28
,다음 페이지로 이동 실패 또는 마지막 페이지: Message: 
,
,크롤링 완료
,총 수집한 리뷰 수: 421


In [8]:
import pandas as pd
df = pd.DataFrame(all_reviews, columns=["review"])

df.to_csv("네이버 스토어 reviews(살로몬)3.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 완료: reviews.csv")

CSV 저장 완료: reviews.csv
